# AHAAR — Ingredient Detector (robust, resumable)

Trains **YOLOv11m** on ~110 pantry classes (36 produce + 19 Indian spices + 55 scraped raw ingredients).

**Design principle: fail fast, never waste a GPU hour.** Every cell 1→6 is < 10 min total. The 1.5–2 hr training cell runs *only* after you can see "110 classes · N images" in the sanity check.

### Before Run All
1. `Runtime → Change runtime type → T4 GPU`
2. Left sidebar 🔑 → add Colab secrets (**Notebook access** on for each):
   - `KAGGLE_USERNAME`, `KAGGLE_KEY`
   - `HF_TOKEN` *(optional, only for upload)*
3. Confirm `scraped.zip` is somewhere in your Drive — the path is configured in cell 2.


## 1. Install (~1 min)

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* kaggle six onnx onnxruntime pyyaml 2>&1 | tail -3
import torch, os, sys
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0), '·',
      round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Config — edit these if needed, then run every cell below in order

In [ ]:
from pathlib import Path

REPO_URL    = 'https://github.com/workspace220106/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'

# Where your scraped.zip lives on Google Drive (file or folder).
DRIVE_SCRAPED_PATH = '/content/drive/MyDrive/AAhar/scraped/scraped.zip'

# Where training outputs land (Drive = survives Colab disconnects)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AAhar/runs'

MODEL     = 'yolo11m.pt'
IMG_SIZE  = 640
EPOCHS    = 60
BATCH     = 16
VAL_SPLIT = 0.10

HF_MODEL_REPO = 'rajivagarwal220106/aahar-ingredient-detector'

print('Config OK.')

## 3. Clone repo + load target class list (~30 s)

In [ ]:
import os, json, subprocess
from google.colab import userdata

os.chdir('/content')
if not Path('/content/aahar').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,
                    '/content/aahar'], check=True)

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
Path('/root/.kaggle').mkdir(exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod('/root/.kaggle/kaggle.json', 0o600)

manifest = json.load(open('/content/aahar/datasets/processed/class_manifest.json'))
TARGET_CLASSES = sorted(set(manifest['ingredient_training_classes']))
print(f'Target ingredient classes: {len(TARGET_CLASSES)}')
assert len(TARGET_CLASSES) == 110, f'Expected 110 target classes, got {len(TARGET_CLASSES)}'

## 4. Mount Drive → extract scraped ZIP intelligently (~2 min)

Handles **any** zip layout:
- zip with files at root → `/content/raw/scraped/<class>/*`
- zip with one `scraped/` wrapper → auto-flattens
- zip with deeper nesting → auto-finds the directory that contains the class folders

In [ ]:
import shutil, zipfile
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

base = Path(DRIVE_SCRAPED_PATH)
# Accept: .zip file, a dir containing .zip, or an extracted folder
if base.is_file() and base.suffix.lower() == '.zip':
    src = base
elif base.is_dir():
    zs = sorted(base.glob('*.zip'))
    src = zs[0] if zs else base
elif Path(str(base) + '.zip').exists():
    src = Path(str(base) + '.zip')
else:
    raise FileNotFoundError(f'Nothing usable at {base}')

RAW_SCRAPED = Path('/content/raw/scraped_in')
if RAW_SCRAPED.exists(): shutil.rmtree(RAW_SCRAPED)
RAW_SCRAPED.mkdir(parents=True)

if src.is_file():
    print(f'Unzipping {src} ({src.stat().st_size/1e6:.0f} MB) → {RAW_SCRAPED}')
    with zipfile.ZipFile(src) as z: z.extractall(RAW_SCRAPED)
else:
    print(f'Copying {src} → {RAW_SCRAPED}')
    shutil.copytree(src, RAW_SCRAPED, dirs_exist_ok=True)

# Find the directory that contains the class folders:
# the directory with the largest number of subdirs that look like class folders.
def score(d: Path) -> int:
    if not d.is_dir(): return 0
    subdirs = [x for x in d.iterdir() if x.is_dir()]
    if not subdirs: return 0
    # 'class-like' = contains >= 1 image file directly inside
    good = sum(1 for sd in subdirs
               if any(p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}
                      for p in sd.iterdir() if p.is_file()))
    return good

best_dir, best_score = RAW_SCRAPED, score(RAW_SCRAPED)
for p in RAW_SCRAPED.rglob('*'):
    if p.is_dir():
        s = score(p)
        if s > best_score:
            best_dir, best_score = p, s

SCRAPED_ROOT = best_dir
print(f'Detected class root: {SCRAPED_ROOT}')
print(f'Class folders found: {best_score}')
assert best_score >= 40, f'Only {best_score} class folders found — zip looks wrong'

## 5. Fetch Kaggle ingredient datasets (~8 min, ~3.3 GB)

In [ ]:
RAW = Path('/content/raw'); RAW.mkdir(exist_ok=True)
DATASETS = [
    ('kritikseth/fruit-and-vegetable-image-recognition', RAW/'fruit-veg'),
    ('tankitrathaur/indian-spices-images-data',          RAW/'indian-spices'),
]
for ref, dst in DATASETS:
    if dst.exists() and any(dst.iterdir()):
        print(f'SKIP (exists): {ref}')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    print(f'--- {ref} ---')
    subprocess.run(['kaggle','datasets','download','-d',ref,'-p',str(dst),'--unzip'],
                   check=True)

subprocess.run(['du','-sh',str(RAW/'fruit-veg'), str(RAW/'indian-spices')], check=True)

## 6. Build YOLO layout (~3 min)

Merges 3 sources against `class_manifest.json`, canonicalises names, drops anything not in the 110 target set, writes YOLO full-frame boxes.

In [ ]:
import re, random, yaml
from PIL import Image
from collections import defaultdict

def canon(s: str) -> str:
    s = s.strip().lower().replace('-','').replace('_',' ')
    s = re.sub(r'\s+',' ', s)
    aliases = {
        'carrots':'carrot','raddish':'radish','sweetpotato':'sweet potato',
        'sweetcorn':'sweet corn','jalepeno':'jalapeno','cinnamom stick':'cinnamon stick',
        'rajma':'rajma bean',
    }
    return aliases.get(s, s)

def canon_snake(s: str) -> str:
    # scraped folder names → snake_case, matches manifest entries like 'toor_dal'
    return s.strip().lower().replace('-','_').replace(' ','_')

target_canon = set()
for name in TARGET_CLASSES:
    target_canon.add(canon(name))
    target_canon.add(canon_snake(name))

pool = defaultdict(list)

# produce (fruit-veg train+val+test)
for split in ('train','validation','test'):
    root = Path(f'/content/raw/fruit-veg/{split}')
    if not root.exists(): continue
    for d in root.iterdir():
        if not d.is_dir(): continue
        name = canon(d.name)
        if name not in target_canon: continue
        for p in d.rglob('*'):
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                pool[name].append(p)

# Indian spices
root = Path('/content/raw/indian-spices/indian-spices-image-data')
if root.exists():
    for d in root.iterdir():
        if not d.is_dir() or d.name.startswith('MetaData'): continue
        name = canon(d.name)
        if name not in target_canon: continue
        for p in d.rglob('*'):
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                pool[name].append(p)

# scraped (55 classes)
for d in SCRAPED_ROOT.iterdir():
    if not d.is_dir(): continue
    name = canon_snake(d.name)
    if name not in target_canon: continue
    for p in d.rglob('*'):
        if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
            pool[name].append(p)

# Build canonical class list in deterministic order
class_names = sorted(pool.keys())
cls_id = {n:i for i,n in enumerate(class_names)}
total = sum(len(v) for v in pool.values())
print(f'Classes kept: {len(class_names)} / {len(TARGET_CLASSES)} target')
print(f'Total images: {total}')
print(f'Classes with 0 images (MISSING from source data):')
missing = sorted(set(TARGET_CLASSES) - {canon_snake(n) for n in class_names} - {canon(n) for n in class_names})
# normalise and show missing
got = set()
for n in class_names: got.update({canon(n), canon_snake(n)})
missing = [t for t in TARGET_CLASSES if canon(t) not in got and canon_snake(t) not in got]
for m in missing: print(f'  • {m}')

DATA = Path('/content/aahar/data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    d = DATA / sub
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

random.seed(42)
for name, imgs in pool.items():
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs)*VAL_SPLIT))
    for i, img in enumerate(imgs):
        split = 'val' if i < n_val else 'train'
        try:
            with Image.open(img) as im: im.verify()
        except Exception:
            continue
        new = f'{cls_id[name]:03d}_{i:06d}{img.suffix.lower()}'
        shutil.copy2(img, DATA / f'images/{split}' / new)
        (DATA / f'labels/{split}' / new.rsplit('.',1)[0]).with_suffix('.txt')\
            .write_text(f'{cls_id[name]} 0.5 0.5 1.0 1.0\n')

cfg_path = Path('/content/aahar/aahar.yaml')
cfg_path.write_text(yaml.safe_dump({
    'path':  str(DATA), 'train': 'images/train', 'val': 'images/val',
    'nc':    len(class_names), 'names': class_names,
}, sort_keys=False))

n_train = len(list((DATA/'images/train').iterdir()))
n_val   = len(list((DATA/'images/val').iterdir()))
print(f'\ntrain: {n_train}  val: {n_val}')

## 7. Sanity check — abort here if anything looks wrong

**Do not run cell 8 until this prints ≥ 100 classes and a healthy image count.**
This cell is zero-cost and catches every failure mode early.

In [ ]:
import yaml, random
cfg = yaml.safe_load(open(cfg_path))
print('=== aahar.yaml ===')
print(' nc:', cfg['nc'])
print(' first 10 names:', cfg['names'][:10])
print(' last  10 names:', cfg['names'][-10:])

# Per-class count
from collections import Counter
counts = Counter()
for f in (DATA/'images/train').iterdir():
    counts[int(f.name.split('_')[0])] += 1
print(f'\nPer-class train counts (min/mean/max): '
      f'{min(counts.values())}/{sum(counts.values())//len(counts)}/{max(counts.values())}')
light = sorted([(n, counts[cls_id[n]]) for n in class_names], key=lambda x:x[1])[:10]
print('10 classes with FEWEST images:')
for n, c in light: print(f'  {n:25s} {c}')

if cfg['nc'] < 100:
    print('\n❌ ABORT: fewer than 100 classes. Do NOT run cell 8. '
          'Fix the data path / zip / manifest and re-run cell 6.')
else:
    print('\n✅ OK — safe to run cell 8 (training).')

## 8. Train (~1.5–2 hrs on T4) — resumable

If Colab disconnects, re-run this cell: ultralytics resumes from the latest `last.pt` in the output directory (which is on Drive, so it survives disconnects).

In [ ]:
from ultralytics import YOLO

drive_out = Path(DRIVE_OUTPUT_DIR); drive_out.mkdir(parents=True, exist_ok=True)
last_pt = drive_out / 'aahar_ingredients_v1' / 'weights' / 'last.pt'

if last_pt.exists():
    print(f'Resuming from {last_pt}')
    model   = YOLO(str(last_pt))
    results = model.train(resume=True)
else:
    model   = YOLO(MODEL)
    results = model.train(
        data   = str(cfg_path),
        epochs = EPOCHS,
        imgsz  = IMG_SIZE,
        batch  = BATCH,
        device = 0,
        project= str(drive_out),         # writes to Drive, not /content
        name   = 'aahar_ingredients_v1',
        patience = 15,
        cache  = 'ram',
        amp    = True,
        save   = True,
        save_period = 5,                  # checkpoint every 5 epochs
    )

best = drive_out / 'aahar_ingredients_v1' / 'weights' / 'best.pt'
print('Best model:', best, '(exists:', best.exists(), ')')

## 9. Export ONNX + TFLite + push to HF (~2 min)

In [ ]:
from ultralytics import YOLO
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',   imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('ONNX :', onnx_path)
print('TFLite:', tflite_path)

try:
    from huggingface_hub import HfApi, create_repo
    token = userdata.get('HF_TOKEN')
    if token:
        create_repo(HF_MODEL_REPO, token=token, exist_ok=True, repo_type='model')
        api = HfApi(token=token)
        for p in [best, Path(onnx_path), Path(tflite_path), cfg_path]:
            api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name,
                            repo_id=HF_MODEL_REPO, token=token)
        print(f'Uploaded → https://huggingface.co/{HF_MODEL_REPO}')
    else:
        print('HF_TOKEN not set; skipping upload.')
except Exception as e:
    print('HF upload failed:', e)